In [1]:
import pandas as pd
import plotly.express as px

# =========================
# 1. Charger les données
# =========================
fichier = r"../data/indicateurs_guelmim_oued_noun_cartoghraphie des établissements.csv"

df = pd.read_csv(fichier, encoding="utf-8")

# =========================
# 2. Nettoyage
# =========================
for col in ["Région", "Tableau", "Indicateur"]:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

df["Valeur"] = pd.to_numeric(df["Valeur"], errors="coerce").fillna(0)

# =========================
# 3. Filtrer la région
# =========================
region = "Guelmim-Oued Noun"
df_region = df[df["Région"] == region].copy()

print("Nombre total de lignes :", df_region.shape[0])

# =========================
# 4. Fonction d’analyse
# =========================
def analyse_par_mots_cles(mots_cles, titre):

    condition = pd.Series(True, index=df_region.index)

    for mot in mots_cles:
        condition &= df_region["Tableau"].str.contains(
            mot,
            case=False,
            na=False,
            regex=False
        )

    data = df_region[condition].copy()

    # Supprimer total
    data = data[
        data["Indicateur"].str.lower() != "total"
    ]

    # Supprimer Total_1 Total_2 ...
    data = data[
        ~data["Indicateur"].str.contains(
            "Total_",
            na=False
        )
    ]

    if data.empty:
        print(f"Aucune donnée pour : {titre}")
        return None

    total = data["Valeur"].sum()

    if total == 0:
        data["Pourcentage"] = 0

    else:
        data["Pourcentage"] = (
            data["Valeur"] / total * 100
        )

    data = data.sort_values(
        by="Valeur",
        ascending=False
    )

    print("\n==============================")
    print("Analyse :", titre)
    print("==============================")

    print(
        data[
            ["Indicateur", "Valeur", "Pourcentage"]
        ].round(2)
    )

    fig = px.bar(
        data,
        x="Indicateur",
        y="Valeur",
        text="Valeur",
        title=titre,
        hover_data=["Pourcentage"]
    )

    fig.update_layout(
        xaxis_tickangle=-45
    )

    fig.show()

    return data

# =========================
# 5. Analyses principales
# =========================
secteur_etab = analyse_par_mots_cles(
    ["établissements", "secteur d’activité"],
    "Établissements par secteur"
)

secteur_emploi = analyse_par_mots_cles(
    ["effectif d'emploi", "secteur d’activité"],
    "Emploi par secteur"
)
    

sous_secteur_emploi = analyse_par_mots_cles(
    ["effictif ", "sous-secteur"],
    "Emploi par sous-secteur"
)

if sous_secteur_emploi is not None:
    total_employes = sous_secteur_emploi["Valeur"].sum()
    print("\n Total des épmlois :", int(total_employes))
    
sous_secteur_femmes = analyse_par_mots_cles(
    ["emploi féminin", "sous-secteur"],
    "Emploi féminin par sous-secteur"
)

# =========================
# 6. Analyse stratégique
# Emploi par établissement
# =========================
if secteur_etab is not None and secteur_emploi is not None:

    comparaison = secteur_etab[["Indicateur", "Valeur"]].rename(
        columns={"Valeur": "Etablissements"}
    )

    emploi = secteur_emploi[["Indicateur", "Valeur"]].rename(
        columns={"Valeur": "Emploi"}
    )

    comparaison = comparaison.merge(
        emploi,
       on="Indicateur",
        how="inner"
    )

    comparaison["Emploi_par_etablissement"] = (
        comparaison["Emploi"] / comparaison["Etablissements"]
    ).round(2)

    comparaison = comparaison.replace([float("inf"), -float("inf")], 0)
    comparaison = comparaison.fillna(0)

    print("\nAnalyse stratégique :")
    print(comparaison)

    fig = px.bar(
        comparaison,
        x="Indicateur",
        y="Emploi_par_etablissement",
        text="Emploi_par_etablissement",
        title="Intensité en emploi par secteur"
    )

    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

df.to_csv(r"../data/outputs/analyse_cartographie_etablissement_emplois.csv", index=False)

Nombre total de lignes : 162

Analyse : Établissements par secteur
                                           Indicateur  Valeur  Pourcentage
2                                            Commerce    8498        28.69
23                                 Commerce de détail    8307        28.05
3                                            Services    3909        13.20
0                                           Industrie    1747         5.90
26                        Hébergement et restauration    1297         4.38
34                       Autres activités de services    1272         4.29
1                                        Construction     656         2.21
22                Travaux de construction spécialisés     640         2.16
17                      Industries Textiles & du Cuir     559         1.89
30                           Services aux entreprises     473         1.60
19  Industries Métalliques, Mécaniques, Electrique...     440         1.49
16                       Industri


Analyse : Emploi par secteur
     Indicateur  Valeur  Pourcentage
5     Industrie    4841        68.41
7      Commerce    1130        15.97
6  Construction    1060        14.98
8      Services      45         0.64


Aucune donnée pour : Emploi par sous-secteur

Analyse : Emploi féminin par sous-secteur
                                           Indicateur  Valeur  Pourcentage
73                                       Enseignement    1007        22.01
65                                 Commerce de détail     938        20.50
58                       Industries Agro-alimentaires     664        14.51
68                        Hébergement et restauration     501        10.95
76                       Autres activités de services     353         7.72
59                      Industries Textiles & du Cuir     235         5.14
72                           Services aux entreprises     204         4.46
70               Activités financières et d'assurance     178         3.89
74                    Santé Humaine et action sociale     147         3.21
60               Industries Chimiques & Parachimiques      77         1.68
66                                   Commerce de gros      59         1.29
61  Industri


Analyse stratégique :
     Indicateur  Etablissements  Emploi  Emploi_par_etablissement
0      Commerce            8498    1130                      0.13
1      Services            3909      45                      0.01
2     Industrie            1747    4841                      2.77
3  Construction             656    1060                      1.62
